# Grayscale String Art (Least Squares)

Preprocess a target image, build the line matrix, solve for line weights, and preview the result.

In [ ]:
import matrix
from preprocess import circle_crop
from PIL import Image
from importlib import reload
import matplotlib.pyplot as plt
import numpy as np

reload(matrix)


## Settings

Edit resolution, peg count, and line width here.

In [ ]:
H = 256
P = 120
LINE_WIDTH = 0.2


## Preprocess target

Crop to a circle on a white background and convert to grayscale.

In [ ]:
target = Image.open('razer/tiger2.jpg')
target = circle_crop(target, H, 32).convert('L')
print(np.array(target).shape)

plt.imshow(target, cmap='gray')
plt.axis('off')
plt.show()


## Precalc + least-squares solve

Build matrix `M` (one column per line) and solve `M @ x ≈ target` for continuous line weights.

~1min runtime on Mac

In [ ]:
M, solved = matrix.full_test(H, P, target, sparse=False, line_width=LINE_WIDTH)


## Thresholded render

Convert continuous weights to on/off lines and redraw with antialiasing.

In [ ]:
data = matrix.render_image(M, solved, H, threshold=0.11, line_width=LINE_WIDTH)
plt.imshow(data, cmap='gray')
plt.axis('off')
plt.show()


## Coefficient histogram

Distribution of solved line weights before thresholding.

In [ ]:
plt.hist(solved.flatten(), color='gray', range=(-1, 1), edgecolor='black', alpha=0.7)
plt.xlabel('Coefficient value')
plt.ylabel('Count')
plt.title('Least-squares weights')
plt.show()


## Exact linear reconstruction

Preview `M @ x` directly (no thresholding) — shows the continuous approximation.

In [ ]:
h = int(M.shape[0] ** 0.5)
exact = matrix.render_exact(M, solved, h)
plt.imshow(exact, cmap='gray')
plt.axis('off')
plt.show()


## Non-negative least squares (NNLS)

Same setup, but constrain weights to `x >= 0` using `scipy.optimize.nnls`.

~16min on Mac - maybe use CUDA if possible

In [ ]:
x_nnls = matrix.solve_nnls(M, np.array(target), H)
print(f'range: {x_nnls.min():.4f} .. {x_nnls.max():.4f}')


## NNLS preview

Compare target, exact NNLS reconstruction, and thresholded line render.

In [ ]:
nnls_exact = matrix.render_exact(M, x_nnls, H)
nnls_thresholded = matrix.render_image(M, x_nnls, H, threshold=0.1, line_width=LINE_WIDTH)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(target, cmap='gray')
axes[0].set_title('Target')
axes[0].axis('off')

axes[1].imshow(nnls_exact, cmap='gray')
axes[1].set_title('NNLS exact reconstruction')
axes[1].axis('off')

axes[2].imshow(nnls_thresholded, cmap='gray')
axes[2].set_title('NNLS thresholded render')
axes[2].axis('off')

plt.tight_layout()
plt.show()


## NNLS coefficient histogram

Distribution of non-negative line weights.

In [ ]:
plt.hist(x_nnls.flatten(), color='gray', range=(0, 1), edgecolor='black', alpha=0.7, bins=100)
plt.xlabel('Coefficient value')
plt.ylabel('Count')
plt.title('NNLS weights')
plt.show()
